# Lab 3: Data Structures Comparison - 10 Minute Lab

## Learning Objectives

By the end of this lab, you will be able to compare PyTorch tensors, NumPy arrays, and pandas DataFrames for time, memory, and accuracy.


## Overview

We'll compare three data structures on two key operations:
1. **Element-wise Addition** - Simple, fast operation
2. **Matrix Multiplication** - Computationally intensive operation

In [7]:
import torch
import numpy as np
import pandas as pd
import time
import psutil
import os

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("=== My Setup ===")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print()


=== My Setup ===
PyTorch: 2.6.0+cpu
NumPy: 1.24.4
Pandas: 1.4.4
CUDA available: False



## Part 1: Create Two Test Matrices

Let's create two matrices A and B to test our operations.

In [24]:
# Create two test matrices
size = 10000  # 1000x1000 matrices
A = np.random.randn(size, size).astype(np.float32)
B = np.random.randn(size, size).astype(np.float32)

print(f"Created two {size}x{size} matrices:")
print(f"Matrix A shape: {A.shape}")
print(f"Matrix B shape: {B.shape}")
print(f"Memory of matrix A: {A.nbytes / 1024**2:.1f} MB")
print(f"Memory of matrix B: {B.nbytes / 1024**2:.1f} MB")
print()

# Show a small sample of the data
print("Sample of Matrix A (first 5x5):")
print(A[:5, :5])
print()
print("Sample of Matrix B (first 5x5):")
print(B[:5, :5])
print()


Created two 10000x10000 matrices:
Matrix A shape: (10000, 10000)
Matrix B shape: (10000, 10000)
Memory of matrix A: 381.5 MB
Memory of matrix B: 381.5 MB

Sample of Matrix A (first 5x5):
[[ 0.16657144 -2.2054162   1.5271108   0.50941586 -1.5336955 ]
 [-0.04715032  0.20067382 -0.2961675  -0.1019396   1.1565512 ]
 [-0.02337721  1.1939201  -0.4530348   0.5065666  -0.03773344]
 [ 0.6172411  -1.8643525   0.22354743  1.9756528   0.0220963 ]
 [-0.10002062  0.5808529   1.5594804   1.2224195   1.3983821 ]]

Sample of Matrix B (first 5x5):
[[ 1.7821409   0.14795785  0.13834797  1.3251534   0.37526226]
 [-1.4882432   0.4029886   0.16494682 -0.38077086 -1.5505607 ]
 [-0.29369584  0.7966198  -0.4750154  -0.6056463   0.6586842 ]
 [ 0.21626443 -0.32484394  1.2002336   0.45826116 -0.75079405]
 [-1.4392303   0.2740826  -0.4246036  -0.3035183   1.0357151 ]]



## Part 2: Convert to Different Data Structures

Now let's convert our matrices to PyTorch tensors, NumPy arrays, and pandas DataFrames.


In [25]:
# Convert matrices to different data structures
print("Converting matrices to different data structures...")

# PyTorch Tensors
A_tensor = torch.tensor(A)
B_tensor = torch.tensor(B)

# NumPy Arrays (already have these)
A_numpy = np.array(A)
B_numpy = np.array(B)

# Pandas DataFrames
A_pandas = pd.DataFrame(A)
B_pandas = pd.DataFrame(B)

# Show data structure properties
print("Data Structure Properties:")
print(f"Tensor A shape: {A_tensor.shape}, dtype: {A_tensor.dtype}")
print(f"NumPy A shape: {A_numpy.shape}, dtype: {A_numpy.dtype}")
print(f"Pandas A shape: {A_pandas.shape}, dtype: {A_pandas.dtypes.iloc[0]}")
print()

# Memory usage comparison
tensor_memory = A_tensor.element_size() * A_tensor.nelement() / 1024**2
numpy_memory = A_numpy.nbytes / 1024**2
pandas_memory = A_pandas.memory_usage(deep=True).sum() / 1024**2

print("Memory Usage (MB) for Matrix A:")
print(f"Tensor: {tensor_memory:.1f} MB")
print(f"NumPy:  {numpy_memory:.1f} MB")
print(f"Pandas: {pandas_memory:.1f} MB")
print()


Converting matrices to different data structures...
Data Structure Properties:
Tensor A shape: torch.Size([10000, 10000]), dtype: torch.float32
NumPy A shape: (10000, 10000), dtype: float32
Pandas A shape: (10000, 10000), dtype: float32

Memory Usage (MB) for Matrix A:
Tensor: 381.5 MB
NumPy:  381.5 MB
Pandas: 381.5 MB



## Part 2: Helper Functions

Let's create functions to measure time, memory, and accuracy.

In [26]:
def measure_operation(operation_name, func, *args):
    """Measure time, memory, and return result"""
    # Measure memory before
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024**2  # MB
    
    # Time the operation
    start_time = time.time()
    result = func(*args)
    end_time = time.time()
    
    # Measure memory after
    mem_after = process.memory_info().rss / 1024**2  # MB
    
    return {
        'operation': operation_name,
        'time': end_time - start_time,
        'memory_used': mem_after - mem_before,
        'result': result
    }

def compare_results(result1, result2, name1, name2):
    """Compare two results for accuracy"""
    if hasattr(result1, 'numpy'):
        result1 = result1.numpy()
    if hasattr(result2, 'values'):
        result2 = result2.values
    
    max_diff = np.max(np.abs(result1 - result2))
    return max_diff < 1e-6, max_diff




## Part 3: Operation 1 - Element-wise Addition

Let's compare A + B across all three data structures.

In [27]:
print("=== Element-wise Addition: A + B ===")

# PyTorch Tensor
tensor_result = measure_operation("PyTorch Tensor", lambda: A_tensor + B_tensor)
print(f"PyTorch: {tensor_result['time']:.4f}s, {tensor_result['memory_used']:.1f}MB")

# NumPy Array
numpy_result = measure_operation("NumPy Array", lambda: A_numpy + B_numpy)
print(f"NumPy:   {numpy_result['time']:.4f}s, {numpy_result['memory_used']:.1f}MB")

# Pandas DataFrame
pandas_result = measure_operation("Pandas DataFrame", lambda: A_pandas + B_pandas)
print(f"Pandas:  {pandas_result['time']:.4f}s, {pandas_result['memory_used']:.1f}MB")

# Check accuracy
tensor_numpy_match, tensor_numpy_diff = compare_results(
    tensor_result['result'], numpy_result['result'], "Tensor", "NumPy")
numpy_pandas_match, numpy_pandas_diff = compare_results(
    numpy_result['result'], pandas_result['result'], "NumPy", "Pandas")

print(f"\nAccuracy Check:")
print(f"Tensor ≈ NumPy: {tensor_numpy_match} (max diff: {tensor_numpy_diff:.2e})")
print(f"NumPy ≈ Pandas: {numpy_pandas_match} (max diff: {numpy_pandas_diff:.2e})")

# Performance summary
times = [tensor_result['time'], numpy_result['time'], pandas_result['time']]
fastest_time = min(times)
print(f"\nFastest: {fastest_time:.4f}s")
#print(f"PyTorch speedup: {fastest_time/tensor_result['time']:.1f}x")
#print(f"NumPy speedup:   {fastest_time/numpy_result['time']:.1f}x")
#print(f"Pandas speedup:  {fastest_time/pandas_result['time']:.1f}x")
print()



=== Element-wise Addition: A + B ===
PyTorch: 1.7470s, 1145.2MB
NumPy:   0.2657s, 381.5MB
Pandas:  1.6262s, 675.7MB

Accuracy Check:
Tensor ≈ NumPy: True (max diff: 0.00e+00)
NumPy ≈ Pandas: True (max diff: 0.00e+00)

Fastest: 0.2657s



## Part 4: Operation 2 - Matrix Multiplication

Now let's compare $A \times  B$ (matrix multiplication) across all three data structures.

In [28]:
print("=== Matrix Multiplication: A @ B ===")

# PyTorch Tensor
tensor_mm_result = measure_operation("PyTorch Tensor", lambda: torch.mm(A_tensor, B_tensor))
print(f"PyTorch: {tensor_mm_result['time']:.4f}s, {tensor_mm_result['memory_used']:.1f}MB")

# NumPy Array
numpy_mm_result = measure_operation("NumPy Array", lambda: np.dot(A_numpy, B_numpy))
print(f"NumPy:   {numpy_mm_result['time']:.4f}s, {numpy_mm_result['memory_used']:.1f}MB")

# Pandas DataFrame (convert to numpy for matrix multiplication)
pandas_mm_result = measure_operation("Pandas DataFrame", lambda: A_pandas.values @ B_pandas.values)
print(f"Pandas:  {pandas_mm_result['time']:.4f}s, {pandas_mm_result['memory_used']:.1f}MB")

# Check accuracy
tensor_numpy_mm_match, tensor_numpy_mm_diff = compare_results(
    tensor_mm_result['result'], numpy_mm_result['result'], "Tensor", "NumPy")
numpy_pandas_mm_match, numpy_pandas_mm_diff = compare_results(
    numpy_mm_result['result'], pandas_mm_result['result'], "NumPy", "Pandas")

print(f"\nAccuracy Check:")
print(f"Tensor ≈ NumPy: {tensor_numpy_mm_match} (max diff: {tensor_numpy_mm_diff:.2e})")
print(f"NumPy ≈ Pandas: {numpy_pandas_mm_match} (max diff: {numpy_pandas_mm_diff:.2e})")

# Performance summary
times_mm = [tensor_mm_result['time'], numpy_mm_result['time'], pandas_mm_result['time']]
fastest_mm_time = min(times_mm)
print(f"\nFastest: {fastest_mm_time:.4f}s")
print(f"PyTorch speedup: {fastest_mm_time/tensor_mm_result['time']:.1f}x")
print(f"NumPy speedup:   {fastest_mm_time/numpy_mm_result['time']:.1f}x")
print(f"Pandas speedup:  {fastest_mm_time/pandas_mm_result['time']:.1f}x")
print()


=== Matrix Multiplication: A @ B ===
PyTorch: 6.3834s, 1163.0MB
NumPy:   7.0001s, 1160.0MB
Pandas:  7.8406s, 718.2MB

Accuracy Check:
Tensor ≈ NumPy: False (max diff: 3.05e-04)
NumPy ≈ Pandas: True (max diff: 0.00e+00)

Fastest: 6.3834s
PyTorch speedup: 1.0x
NumPy speedup:   0.9x
Pandas speedup:  0.8x



## Part 5: Summary Table

Let's create a clear summary of our results.

In [33]:
# Create summary table
summary_data = {
    'Data Structure': ['PyTorch Tensor', 'NumPy Array', 'Pandas DataFrame'],
    'Addition Time (s)': [f"{tensor_result['time']:.4f}", f"{numpy_result['time']:.4f}", f"{pandas_result['time']:.4f}"],
    'Addition Memory (MB)': [f"{tensor_result['memory_used']:.1f}", f"{numpy_result['memory_used']:.1f}", f"{pandas_result['memory_used']:.1f}"],
    'Matrix Mult Time (s)': [f"{tensor_mm_result['time']:.4f}", f"{numpy_mm_result['time']:.4f}", f"{pandas_mm_result['time']:.4f}"],
    'Matrix Mult Memory (MB)': [f"{tensor_mm_result['memory_used']:.1f}", f"{numpy_mm_result['memory_used']:.1f}", f"{pandas_mm_result['memory_used']:.1f}"],
    'Accuracy': ['?', '?', '?']
}
print(pd.DataFrame(summary_data))



     Data Structure Addition Time (s) Addition Memory (MB)  \
0    PyTorch Tensor            1.7470               1145.2   
1       NumPy Array            0.2657                381.5   
2  Pandas DataFrame            1.6262                675.7   

  Matrix Mult Time (s) Matrix Mult Memory (MB) Accuracy  
0               6.3834                  1163.0        ?  
1               7.0001                  1160.0        ?  
2               7.8406                   718.2        ?  


## Bonus: Quick GPU Test (if available)

Let's see if GPU acceleration makes a difference!

In [23]:
if torch.cuda.is_available():
    print("GPU detected! Testing GPU vs CPU...")
    
    # Create larger matrices for GPU test
    size_gpu = 1500
    A_gpu = np.random.randn(size_gpu, size_gpu).astype(np.float32)
    B_gpu = np.random.randn(size_gpu, size_gpu).astype(np.float32)
    
    # CPU tensor
    A_cpu = torch.tensor(A_gpu)
    B_cpu = torch.tensor(B_gpu)
    
    # GPU tensor
    A_gpu_tensor = A_cpu.cuda()
    B_gpu_tensor = B_cpu.cuda()
    
    # CPU matrix multiplication
    cpu_time = time.time()
    result_cpu = torch.mm(A_cpu, B_cpu)
    cpu_time = time.time() - cpu_time
    
    # GPU matrix multiplication
    gpu_time = time.time()
    result_gpu = torch.mm(A_gpu_tensor, B_gpu_tensor)
    torch.cuda.synchronize()
    gpu_time = time.time() - gpu_time
    
    print(f"Matrix size: {size_gpu}x{size_gpu}")
    print(f"CPU time: {cpu_time:.4f}s")
    print(f"GPU time: {gpu_time:.4f}s")
    print(f"GPU speedup: {cpu_time/gpu_time:.1f}x faster!")
    print(f"Results equivalent: {torch.allclose(result_cpu, result_gpu.cpu())}")
else:
    print("No GPU available - install CUDA PyTorch for GPU acceleration")


No GPU available - install CUDA PyTorch for GPU acceleration


# A Second Bonus
Polar?

In [ ]:
import polars as pl

# Add Polars comparison
A_polars = pl.DataFrame(A)
B_polars = pl.DataFrame(B)
polars_add_result = measure_operation("Polars DataFrame Addition", lambda: A_polars + B_polars)
# Polars does not natively support matrix multiplication (the @ operator) on DataFrames.
# Therefore, we must convert to numpy for the matrix multiplication.
# If you want to keep everything in Polars, you would need to implement matrix multiplication manually,
# but Polars DataFrames are not designed for this operation.
polars_mul_result = measure_operation(
    "Polars DataFrame Multiplication",
    lambda: pl.DataFrame(A_polars.to_numpy() @ B_polars.to_numpy())
)

# Create summary table including Polars results
summary_data = {
    'Data Structure': ['PyTorch Tensor', 'NumPy Array', 'Pandas DataFrame', 'Polars DataFrame'],
    'Addition Time (s)': [
        f"{tensor_result['time']:.4f}",
        f"{numpy_result['time']:.4f}",
        f"{pandas_result['time']:.4f}",
        f"{polars_add_result['time']:.4f}"
    ],
    'Addition Memory (MB)': [
        f"{tensor_result['memory_used']:.1f}",
        f"{numpy_result['memory_used']:.1f}",
        f"{pandas_result['memory_used']:.1f}",
        f"{polars_add_result['memory_used']:.1f}"
    ],
    'Matrix Mult Time (s)': [
        f"{tensor_mm_result['time']:.4f}",
        f"{numpy_mm_result['time']:.4f}",
        f"{pandas_mm_result['time']:.4f}",
        f"{polars_mul_result['time']:.4f}"
    ],
    'Matrix Mult Memory (MB)': [
        f"{tensor_mm_result['memory_used']:.1f}",
        f"{numpy_mm_result['memory_used']:.1f}",
        f"{pandas_mm_result['memory_used']:.1f}",
        f"{polars_mul_result['memory_used']:.1f}"
    ],
    'Accuracy': ['?', '?', '?', '?']
}
print(pd.DataFrame(summary_data))


     Data Structure Addition Time (s) Addition Memory (MB)  \
0    PyTorch Tensor            1.7470               1145.2   
1       NumPy Array            0.2657                381.5   
2  Pandas DataFrame            1.6262                675.7   
3  Polars DataFrame            0.1222                399.7   

  Matrix Mult Time (s) Matrix Mult Memory (MB) Accuracy  
0               6.3834                  1163.0        ?  
1               7.0001                  1160.0        ?  
2               7.8406                   718.2        ?  
3               7.6505                    56.5        ?  
